# 01 · 가설·지표·고정 배정

**핵심 실행 8분** · 실험을 시작하기 전에 바꿀 것 하나, 분석 단위, 종료 기준을 정합니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 실험 계약 읽기 | A/B 50:50, 최소 각 1000명, 14일, 관찰 창 30분, p95 상한 1800ms. |
| 3 | 변경 하나와 가설 | 변경=답변 구성, 단위=첫 진입 사용자, 지표=30분 내 해결 확인율. |
| 4 | 동일 사용자 재방문 | u-alpha와 u-beta는 현재 설정에서 모두 B. 각 ID의 4회 방문이 같은 버전이며 assert 통과. |
| 5 | 2400명 배정 확인 | A 1196명 / B 1204명, SRM p≈0.87028. |
| 6 | 실험 중 배정 비율 변경 | B 비율을 0.5→0.2로 바꾸면 686명의 버전과 설정 해시가 바뀜. |
| 7 | 요청 수와 사용자 수의 차이 | A 사용자 1196 / 요청 3054, B 사용자 1204 / 요청 2972. |

**이번 파일의 역할:** 누구에게 무엇을 비교할지 고정하는 실험 설계. 10주차의 실행 설정에 사용자 배정·분석 단위·판정 시점을 추가합니다.

**재실행 방법:** 전체 재실행 가능. 셀 6은 changed라는 별도 객체를 사용하므로 DEFAULT나 원본 CSV를 바꾸지 않습니다.

**코드 셀 1 설명:** 실험을 비교하기 전에 실행 환경과 조건부터 맞춥니다. 파일별 커널은 독립적이므로 각 Notebook의 첫 코드 셀부터 실행합니다.

번호는 코드 셀 순서이며 실행 횟수 `[1]`, `[2]`와 다릅니다. [셀별 교수 설명](https://smilesjcha.github.io/llmops-16week/week12/lecture/instructor/#book-01)

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 실험 계약 읽기

성과를 보기 전에 정한 기준입니다. 이 숫자는 수업용 운영 기준이며 보편적인 산업 표준이 아닙니다.

In [ ]:
from dataclasses import asdict, replace
display(pd.DataFrame(asdict(DEFAULT).items(), columns=["설정", "값"]))

**확인과 해석:** A/B 50:50, 최소 각 1000명, 14일, 관찰 창 30분, p95 상한 1800ms.

**화면에서 볼 곳:** allocation_b·planned_days·observation_minutes·max_p95_ms

**설명 핵심:** 결과를 보기 전에 성공 기준과 중단 기준을 정합니다. 1000명은 수업 계약이며 모든 3%p 개선을 충분한 검정력으로 탐지한다는 뜻은 아닙니다.

**직접 설명:** p95 기준을 결과 확인 뒤 높이면 어떤 문제가 생길까요?

## 코드 셀 3 · 변경 하나와 가설

A는 짧은 직접 답변, B는 조건·비용·다음 행동입니다. 모델·정책 원문·대상·기능은 같게 둡니다.

In [ ]:
hypothesis = {"change": "답변 구성 A → B", "unit": "첫 실험 진입 사용자", "primary": "30분 내 해결 확인 사용자 / 진입 사용자", "guardrails": "p95 ≤ 1800ms, 오류율 ≤ 3%", "expected_reason": "다음 행동 이해 개선"}
display(pd.Series(hypothesis))

**확인과 해석:** 변경=답변 구성, 단위=첫 진입 사용자, 지표=30분 내 해결 확인율.

**화면에서 볼 곳:** change·unit·primary·expected_reason

**설명 핵심:** 다음 행동을 이해하기 쉽다는 설명 가설과 실제로 관찰하는 해결 확인 지표를 구분합니다. 모델까지 동시에 바꾸면 답변 구성만의 효과라고 설명하기 어렵습니다.

**직접 설명:** 모델과 프롬프트를 동시에 바꾸면 어떤 변경의 효과인지 알 수 있나요?

## 코드 셀 4 · 동일 사용자 재방문

Python의 기본 hash는 실행 간 고정되지 않을 수 있습니다. 실습은 SHA-256 해시로 배정합니다.

In [ ]:
visits = pd.DataFrame({"visitor_id": ["u-alpha"]*4 + ["u-beta"]*4})
visits["variant"] = visits.visitor_id.map(assign)
display(visits)
assert visits.groupby("visitor_id").variant.nunique().max() == 1

**확인과 해석:** u-alpha와 u-beta는 현재 설정에서 모두 B. 각 ID의 4회 방문이 같은 버전이며 assert 통과.

**화면에서 볼 곳:** 같은 visitor_id의 variant 4개

**설명 핵심:** 사용자마다 반드시 서로 다른 버전을 받는 것은 아닙니다. 두 명 모두 B여도 정상이며, 여기서 확인하는 것은 같은 사람의 경험이 유지되는지입니다.

**직접 설명:** 요청할 때마다 동전을 던지면 같은 사람의 경험이 어떻게 달라질까요?

## 코드 셀 5 · 2400명 배정 확인

개별 사용자의 버전이 안정적이어도 전체 비율은 정확히 1200:1200일 필요가 없습니다.

In [ ]:
users = pd.DataFrame({"visitor_id": [f"u-{i:05d}" for i in range(2400)]})
users["variant"] = users.visitor_id.map(assign)
display(users.variant.value_counts().rename("사용자 수"))
from week12.lab.analysis import srm_test
counts = users.variant.value_counts()
print(srm_test(int(counts["A"]), int(counts["B"])))

**확인과 해석:** A 1196명 / B 1204명, SRM p≈0.87028.

**화면에서 볼 곳:** 사용자 수와 p_value

**설명 핵심:** 반반 배정은 정확히 같은 인원수를 보장하지 않습니다. 이 차이는 기대 비율과 크게 어긋나지 않지만, p가 높다고 모든 수집 과정이 정확하다고 증명된 것은 아닙니다.

**직접 설명:** 8명 차이가 있다는 사실만으로 버그라고 판단해도 될까요?

## 코드 셀 6 · 실험 중 배정 비율 변경

설정을 50:50에서 80:20으로 바꾸면 같은 사용자도 다른 버전이 될 수 있습니다. 수업 구현은 실험 계약을 고정합니다.

In [ ]:
changed = replace(DEFAULT, allocation_b=.2)
users["new_variant"] = users.visitor_id.map(lambda u: assign(u, changed))
print("버전 변경 사용자:", int((users.variant != users.new_variant).sum()))
print("기존/변경 설정:", DEFAULT.version, changed.version)

**확인과 해석:** B 비율을 0.5→0.2로 바꾸면 686명의 버전과 설정 해시가 바뀜.

**화면에서 볼 곳:** 버전 변경 사용자와 기존/변경 설정

**설명 핵심:** 같은 실험 도중 비율을 바꾸면 기존 사용자의 경험과 해석 조건이 달라집니다. 실험에 진입시키는 전체 트래픽 비율과 내부 A/B 비율을 별도로 관리해야 합니다.

**직접 설명:** 점진 배포 트래픽 비율과 실험 내부의 A/B 비율을 왜 분리할까요?

## 코드 셀 7 · 요청 수와 사용자 수의 차이

한 사용자의 반복 요청은 독립 사용자를 추가하지 않습니다. metric의 분모를 미리 고정합니다.

In [ ]:
data = load_units("slow")
display(data.groupby("variant").agg(users=("visitor_id","nunique"), requests=("requests","sum")))
print("분석은 사용자 한 명당 첫 요청 결과 한 건입니다.")

**확인과 해석:** A 사용자 1196 / 요청 3054, B 사용자 1204 / 요청 2972.

**화면에서 볼 곳:** users와 requests 열

**설명 핵심:** 10주차는 처리 요청을 집계했고 이번 주는 독립 사용자 결과를 비교합니다. 요청 6026건을 사용자 6026명처럼 계산하면 같은 사람을 여러 표로 세게 됩니다.

**직접 설명:** 요청이 많은 몇 명에게 결론이 좌우되지 않으려면 어떻게 집계할까요?

## 복습과 다음 단계

종이에 실험 가설을 4줄로 적고, 실험 ID·설정 버전을 바꾸어 배정이 달라지는지 복습합니다. 다음은 실제 API에서 이 계약을 지키는지 확인합니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
